# 05 E5: shortcuts, mitigation (and later calibration)

All models are the final ones (chosen on dev, notebooks 02-03), refit from code with seed 0.
Test-time changes (`deepvoice.features.apply_transform`): **trim** (cut leading/trailing silence
with the group-S VAD), **gain-12** (12 dB quieter), **gain+6** (6 dB louder, clipped). Features of
changed audio are cached as `<group>_<split>_<change>.parquet`.

- **H1b** (frozen): trimming silence raises the rung 1 eval EER by >= 5 pp with the paired
  speaker-bootstrap CI excluding 0.
- XLS-R normalizes every clip to zero mean and unit variance, so gain changes cannot change its
  input (apart from clipped samples at +6 dB); it gets the trim test only.
- **Mitigation (E5.2):** rung 1 retrained on augmented train audio (trim, random 0-1 s noise padding,
  random -12..+6 dB gain; seeded per clip), same hyperparameters, compared on the same clips.

Results: `results/e5.csv` (standard columns; `split` names the change) and
`results/e5_diffs.csv` (paired differences with 95% CIs).

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from deepvoice.data import load_itw
from deepvoice.features import feature_tables, load_xy
from deepvoice.metrics import paired_speaker_bootstrap_diff
from deepvoice.models import run

# cached features of changed audio (computed once; loaded here)
p = pd.read_parquet("data/protocols.parquet")
sets = {"eval": p[p.split == "eval"].reset_index(drop=True), "itw": load_itw().reset_index(drop=True)}
for change in ["trim", "gain-12", "gain+6"]:
    for s, df in sets.items():
        feature_tables(df, f"{s}_{change}", transform=change)
feature_tables(p[p.split == "train"].reset_index(drop=True), "train_augment", transform="augment")

FS = {"groups": ("lfcc",), "n_coef": 40, "deltas": 2, "stats": ("mean", "std")}
gap = pd.read_csv(ROOT / "results/e3_gap.csv")
HGB = json.loads(gap[(gap.model == "hgb") & (gap["selected by"] == "dev")].params.iloc[0])
EV = {"dev": load_xy("dev", **FS)}
for s in ["eval", "itw"]:
    EV[s] = load_xy(s, **FS)
    for change in ["trim", "gain-12", "gain+6"]:
        EV[f"{s}_{change}"] = load_xy(s, variant=change, **FS)
train, train_aug = load_xy("train", **FS), load_xy("train", variant="augment", **FS)
X = list(train.columns[4:])
DATASET = lambda s: "itw" if s.startswith("itw") else "asvspoof"
print({k: len(v) for k, v in EV.items()})

## Rung 1: original vs augmented training, on clean and changed audio

In [ ]:
models, scores = {}, {}
for tag, tr in [("lfcc40dd_e3devsel", train), ("lfcc40dd_e3devsel_augment", train_aug)]:
    m, _ = run(1, "hgb", X, tr, [(DATASET(s), s, EV[s]) for s in EV], params=HGB,
               experiment="e5", feature_set=tag)
    models[tag] = m
    scores[tag] = {s: m.predict_proba(EV[s][X].to_numpy())[:, 1] for s in EV}
e5 = pd.read_csv(ROOT / "results/e5.csv")
e5[e5.rung == 1].assign(eer=lambda d: (100 * d.eer).round(2)).pivot_table(index="split", columns="model", values="eer")

## Paired differences (EER of B minus EER of A on the same clips, 95% CI)

In [ ]:
rows = []
orig, aug = scores["lfcc40dd_e3devsel"], scores["lfcc40dd_e3devsel_augment"]
for s in ["eval", "itw"]:
    for change in ["trim", "gain-12", "gain+6"]:  # same clips and labels, changed audio
        d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, orig[s], orig[f"{s}_{change}"], EV[s].speaker)
        rows.append({"model": "rung 1 (original)", "A": f"{s} clean", "B": f"{s} {change}", "clips": s,
                     "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
    for v in ["", "_trim"]:
        d, lo, hi = paired_speaker_bootstrap_diff(EV[s].label, orig[s + v], aug[s + v], EV[s].speaker)
        rows.append({"model": "augmented minus original", "A": f"original on {s}{v.replace('_', ' ')}",
                     "B": f"augmented on {s}{v.replace('_', ' ')}", "clips": s,
                     "diff_pp": 100 * d, "ci_low_pp": 100 * lo, "ci_high_pp": 100 * hi})
diffs = pd.DataFrame(rows)
diffs.to_csv(ROOT / "results/e5_diffs.csv", index=False, float_format="%.3f")
diffs.round(2)

## H1b verdict (rule from the frozen `hypotheses.md`)

In [ ]:
h = diffs[(diffs.model == "rung 1 (original)") & (diffs.B == "eval trim")].iloc[0]
h1b = h.diff_pp >= 5 and h.ci_low_pp > 0
print(f"Trimming raises rung 1 eval EER by {h.diff_pp:.1f} pp [{h.ci_low_pp:.1f}, {h.ci_high_pp:.1f}] -> H1b {'TRUE' if h1b else 'FALSE'}")